In [2]:
import json
import openai
import requests

client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [4]:
# 함수 정의

def get_popular_movies():
    """인기 영화 목록을 가져온다."""
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    """영화 상세 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}").json()

def get_movie_credits(id):
    """영화 출연진/제작진 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}/credits").json()

# 함수 매핑
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [5]:
# AI에게 알려줄 도구(tools) 정의
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [6]:
# 대화 이력 (메모리)
messages = []
messages.append(
    {
        "role": "system",
        "content": "You are a movie expert agent. Answer questions about movies using the provided tools. Reply in the same language the user uses.",
    }
)

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    message = response.choices[0].message
    process_ai_response(message)

def process_ai_response(message):
    if message.tool_calls:
        # AI의 tool_calls 응답을 messages에 기록
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ],
            }
        )
        # 각 tool_call에 대해 실제 함수 실행
        for tc in message.tool_calls:
            fn_name = tc.function.name
            arguments = tc.function.arguments
            try:
                args = json.loads(arguments)
            except json.JSONDecodeError:
                args = {}
            print(f"Function Calling {fn_name}({args})")

            function_to_run = FUNCTION_MAP.get(fn_name)
            result = function_to_run(**args)

            # 실행 결과를 role: "tool"로 추가
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )
        # 도구 결과를 포함해서 AI 다시 호출
        call_ai()
    else:
        # 일반 텍스트 응답
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

In [7]:
# 테스트 1: 인기 영화
messages.append({"role": "user", "content": "지금 인기 있는 영화가 무엇인지 알려줘"})
call_ai()

Function Calling get_popular_movies({})
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Your Heart Will Be Broken**
   - 개요: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하다가 주범인 바르스와 연인 행세를 하기로 계약을 맺는다. 그 과정에서 진정한 감정이 싹트지만, 그녀의 가족과 동급생들은 이들을 갈라놓으려 한다.
   - 개봉일: 2026년 3월 26일
   - 평점: 6.6
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - 개요: 제이크 설리와 네이티리가 판도라에서 새로운 위협과 맞서 싸우며 가족을 지키기 위한 여정을 그린 이야기.
   - 개봉일: 2025년 12월 17일
   - 평점: 7.4
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - 개요: 사람의 의식을 로봇 동물로 보내 대화할 수 있는 기술을 발견한 과학자들이 그 과정에서 동물의 세계의 비밀을 파헤치는 이야기.
   - 개봉일: 2026년 3월 4일
   - 평점: 7.6
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Crime 101**
   - 개요: 고양이 도둑과 보험 중개인이 얽힌 이야기를 다룬 범죄 thriller.
   - 개봉일: 2026년 2월 11일
   - 평점: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w780/tVvpFIoteRHNnoZMhdnwIVwJpCA.jpg)

5. **Mike & Nick & Nick & Alice**
   - 개요: 두 갱스터와 그들이 사랑하는 여성이 가장 위험한

In [8]:
# 테스트 2: 영화 상세 정보
messages.append({"role": "user", "content": "movie ID 550에 해당하는 영화가 무엇인지 알려줘"})
call_ai()

Function Calling get_movie_details({'id': 550})
AI: 영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다.

- **개요**: 한 시한폭탄처럼 불안한 불면증 환자와 미끄러운 비누 판매자가 원초적인 남성의 공격성을 충동적으로 새로운 형태의 치료로 채널링한다. 이들의 개념은 퍼지면서 각 마을에서 지하 "격투 클럽"이 생기게 되고, 한 기이한 인물이 그들을 방해하여 휘말리게 되는 이야기를 담고 있습니다.
- **개봉일**: 1999년 10월 15일
- **러닝타임**: 139분
- **장르**: 드라마, 스릴러
- **평점**: 8.4
- **제작사**: 
  - Fox 2000 Pictures
  - Regency Enterprises
  - Linson Entertainment
  - 20th Century Fox
  - Taurus Film
- **예산**: 63,000,000달러
- **수익**: 100,853,753달러
- **태그라인**: Mischief. Mayhem. Soap.
- **홈페이지**: [Fight Club](https://www.20thcenturystudios.com/movies/fight-club)
- **포스터**: 
![포스터](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)

더 알고 싶은 내용이 있으면 말씀해 주세요!


In [9]:
# 테스트 3: 출연진 정보
messages.append({"role": "user", "content": "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"})
call_ai()

Function Calling get_movie_credits({'id': 550})
AI: **"Fight Club"**에 출연한 주요 배우들은 다음과 같습니다:

1. **Edward Norton** - Character: Narrator
   - ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **Brad Pitt** - Character: Tyler Durden
   - ![Brad Pitt](https://image.tmdb.org/t/p/w185/r9DzKQLNbh5QfXlrFGHoVNKER7X.jpg)

3. **Helena Bonham Carter** - Character: Marla Singer
   - ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **Meat Loaf** - Character: Robert Paulson
   - ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)

5. **Jared Leto** - Character: Angel Face
   - ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

6. **Zach Grenier** - Character: Richard Chesler (Regional Manager)
   - ![Zach Grenier](https://image.tmdb.org/t/p/w185/fSyQKZO39sUsqY283GXiScOg3Hi.jpg)

7. **Holt McCallany** - Character: The Mechanic
   - ![Holt McCallany](https://image.tmdb